# Train and evaluate — Unsloth LoRA (mock path for CI)

Uses `benchmarks/training/toy_feedback.jsonl` and **stub** backend by default.
Set `USE_UNSLOTH = True` for GPU training with `pip install -e '.[unsloth]'`.

In [ ]:
import asyncio
import os
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIXTURES = REPO / "benchmarks" / "training" / "toy_feedback.jsonl"
USE_UNSLOTH = False
MODEL_ID = "qwen3-0.6b"
RUN_ID = "notebook-smoke"

os.environ.setdefault("RADA_ADAPTER_STORE_ROOT", str(REPO / ".cache" / "adapters"))

from rada.training.config import TrainingConfig
from rada.training.dataset import load_training_dataset
from rada.training.unsloth_trainer import build_trainer
from rada.backends import StubLLMBackend

backend_name = "unsloth" if USE_UNSLOTH else "stub"
config = TrainingConfig(
    model_id=MODEL_ID,
    backend=backend_name,
    data_source="export",
    data_path=FIXTURES,
    output_run_id=RUN_ID,
    epochs=1,
)
examples = load_training_dataset("export", data_path=FIXTURES)
artifact = build_trainer(config).train(examples)
print(artifact)

In [ ]:
backend = StubLLMBackend(model_id=MODEL_ID).with_lora(artifact.adapter_path)
completion = asyncio.run(backend.complete("Evaluate BTCUSD hold rationale"))
completion

## Pre vs post comparison (optional)

In [ ]:
from rada.evaluation.pre_post_compare import run_pre_post_compare

report = asyncio.run(
    run_pre_post_compare(
        MODEL_ID,
        FIXTURES,
        adapter_path=artifact.adapter_path,
        train=False,
    )
)
report.to_dict()